## Prerequisites

- A Python kernel that has `rpy2`, `pandas`, and `rpy-bridge` installed.
- R must be installed and available on `PATH` (rpy2 needs R).
- This repo should contain `examples/toy_funcs.R` (we added it to `examples/`).

If you are missing R or rpy2, install them before running these cells. In CI you must install R (we added `r-lib/actions/setup-r` to the workflow).

In [1]:
# Environment checks: verify Python dependencies and R availability
import sys
from pathlib import Path

print("Python", sys.version.splitlines()[0])

try:
    import rpy2  # noqa: F401

    print("rpy2: OK")
except Exception as e:
    print("rpy2 import failed:", e)

import shutil  # noqa: E402
import subprocess  # noqa: E402

R_cmd = shutil.which("R")
if R_cmd:
    try:
        out = subprocess.run([R_cmd, "--version"], capture_output=True, text=True, timeout=5)
        print("R available:", out.stdout.splitlines()[0])
    except Exception as e:
        print("R found but --version failed:", e)
else:
    print("R not found on PATH; rpy2 will not work until R is installed")

# Check example script path
script_path = Path("./toy_funcs.R")
print("example script exists:", script_path.exists(), script_path)

Python 3.11.11 (main, Dec  6 2024, 21:09:50) [Clang 18.1.8 ]
rpy2: OK
R available: R version 4.5.2 (2025-10-31) -- "[Not] Part in a Rumble"
example script exists: True toy_funcs.R


## Run the local example script

This uses the local `examples/toy_funcs.R` file. The R functions are simple and safe: `add_and_scale` and `multiply_table`.

In [ ]:
from pathlib import Path

from rpy_bridge import RFunctionCaller

script = Path("./toy_funcs.R")
rfc = RFunctionCaller(path_to_renv=None, scripts=script)

print("add_and_scale(2,3) ->")
print(rfc.call("add_and_scale", 2, 3))

print("add_and_scale(2,3, scale=10) ->")
print(rfc.call("add_and_scale", 2, 3, scale=10))

print("multiply_table(2,5,times=4) -> pandas DataFrame")
df = rfc.call("multiply_table", 2, 5, times=4)
print(type(df))
print(df.head())

TypeError: RFunctionCaller.__init__() received unexpected keyword arguments: ['script']

## Using `renv`

If your R project uses `renv` to manage packages, pass the project directory (the parent that contains `renv/`) as `path_to_renv` when creating the caller. `RFunctionCaller` will call `renv::load()` before sourcing the script so the R session uses the project's library versions.

Only run the next cell if you have a local project with `renv` restored.

In [ ]:
# Example showing how to pass a local renv project (commented out by default)
from pathlib import Path

# local_project should be the folder that contains renv/activate.R and renv.lock
local_project = Path("/path/to/local/project")  # <-- change this if you have an renv
# script_path = local_project / 'examples' / 'toy_funcs.R'
# caller = RFunctionCaller(path_to_renv=local_project, script=script_path)
# print(caller.call('add_and_scale', 1, 2, scale=3))

print("Example shows how to supply path_to_renv; commented out to avoid accidental execution")

## How Python args map to R

- `caller.call('f', 1, 2)` -> R `f(1, 2)`
- `caller.call('f', a=1, b=2)` -> R `f(a = 1, b = 2)`
- Mix positional then named: `caller.call('f', 1, b=2)` -> `f(1, b = 2)`

Return mapping:
- R `data.frame` -> pandas `DataFrame`
- R named `list` -> Python `dict`
- Scalars -> Python `int/float`

## Next steps / exercises

- Modify `examples/toy_funcs.R` to return a summary `data.frame` and call it from Python.
- Try the GitHub inspect flow on a trusted repository.
- Add a small `renv` to a test project and pass `path_to_renv` to the caller.